# Day 4 — PySpark Practice Questions: Aggregations

| # | Difficulty | Topic |
|---|-----------|-------|
| 1 | 🟢 Easy    | Total revenue (quantity × unit_price) per region |
| 2 | 🟢 Easy    | Products with more than 50 total units sold |
| 3 | 🟡 Medium  | Per region: min price, max price, avg price, total qty |

In [1]:
import os
os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Day4_PySpark_PQ') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

# Shared dataset — same as SQL notebook
sales_data = [
    ('S001','P001','North','2024-01-05', 10, 29.99),
    ('S002','P002','South','2024-01-07',  5, 49.99),
    ('S003','P001','East', '2024-01-10', 20, 29.99),
    ('S004','P003','West', '2024-01-12', 15, 99.99),
    ('S005','P002','North','2024-01-15',  8, 49.99),
    ('S006','P001','South','2024-02-01', 30, 29.99),
    ('S007','P003','East', '2024-02-03',  2, 99.99),
    ('S008','P004','West', '2024-02-08', 25, 14.99),
    ('S009','P004','North','2024-02-11', 40, 14.99),
    ('S010','P002','East', '2024-02-14', 12, 49.99),
    ('S011','P001','West', '2024-03-01', 18, 29.99),
    ('S012','P003','South','2024-03-05',  7, 99.99),
    ('S013','P004','East', '2024-03-09', 22, 14.99),
    ('S014','P002','West', '2024-03-12',  3, 49.99),
    ('S015','P001','North','2024-03-20', 14, 29.99),
]

schema = StructType([
    StructField('sale_id',    StringType(),  False),
    StructField('product_id', StringType(),  True),
    StructField('region',     StringType(),  True),
    StructField('sale_date',  StringType(),  True),
    StructField('quantity',   IntegerType(), True),
    StructField('unit_price', DoubleType(),  True),
])

df_sales = spark.createDataFrame(sales_data, schema=schema)
df_sales.printSchema()
df_sales.show(5)
print('Total rows:', df_sales.count())

Spark version: 3.5.6
root
 |-- sale_id: string (nullable = false)
 |-- product_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)

+-------+----------+------+----------+--------+----------+
|sale_id|product_id|region| sale_date|quantity|unit_price|
+-------+----------+------+----------+--------+----------+
|   S001|      P001| North|2024-01-05|      10|     29.99|
|   S002|      P002| South|2024-01-07|       5|     49.99|
|   S003|      P001|  East|2024-01-10|      20|     29.99|
|   S004|      P003|  West|2024-01-12|      15|     99.99|
|   S005|      P002| North|2024-01-15|       8|     49.99|
+-------+----------+------+----------+--------+----------+
only showing top 5 rows

Total rows: 15


---
# Question 1 🟢 — Total Revenue per Region

## Concept Warm-up

In [2]:
# Warm-up 1: withColumn to create a computed column
df_sales.withColumn(
    'revenue', F.col('quantity') * F.col('unit_price')
).select('sale_id', 'region', 'quantity', 'unit_price', 'revenue').show(5)

+-------+------+--------+----------+------------------+
|sale_id|region|quantity|unit_price|           revenue|
+-------+------+--------+----------+------------------+
|   S001| North|      10|     29.99|             299.9|
|   S002| South|       5|     49.99|249.95000000000002|
|   S003|  East|      20|     29.99|             599.8|
|   S004|  West|      15|     99.99|           1499.85|
|   S005| North|       8|     49.99|            399.92|
+-------+------+--------+----------+------------------+
only showing top 5 rows



In [3]:
# Warm-up 2: groupBy + F.sum on the revenue column
(
    df_sales
    .withColumn('revenue', F.col('quantity') * F.col('unit_price'))
    .groupBy('region')
    .agg(F.sum('revenue').alias('total_revenue_raw'))
    .show()
)

+------+------------------+
|region| total_revenue_raw|
+------+------------------+
| North|           1719.28|
| South|           1849.58|
|  East|1729.4399999999998|
|  West|           2564.39|
+------+------------------+



In [4]:
# Warm-up 3: F.round to round to 2 decimal places
(
    df_sales
    .withColumn('revenue', F.col('quantity') * F.col('unit_price'))
    .groupBy('region')
    .agg(F.round(F.sum('revenue'), 2).alias('total_revenue'))
    .show()
)

+------+-------------+
|region|total_revenue|
+------+-------------+
| North|      1719.28|
| South|      1849.58|
|  East|      1729.44|
|  West|      2564.39|
+------+-------------+



## Problem

Compute total revenue (`quantity × unit_price`) per region.  
Sort by `total_revenue` **descending** — highest revenue first.

**Expected columns:** `region`, `total_revenue`

In [ ]:
# YOUR CODE HERE
df_revenue = None

# df_revenue.show()

---
# Question 2 🟢 — Products with More Than 50 Units Sold

## Concept Warm-up

In [ ]:
# Warm-up 1: total units per product — no filter yet
(
    df_sales
    .groupBy('product_id')
    .agg(F.sum('quantity').alias('total_units'))
    .orderBy('total_units', ascending=False)
    .show()
)

In [ ]:
# Warm-up 2: .filter() after .agg() = SQL HAVING
# Filter threshold = 30 to see what the pattern looks like
(
    df_sales
    .groupBy('product_id')
    .agg(F.sum('quantity').alias('total_units'))
    .filter(F.col('total_units') > 30)        # ← THIS is HAVING
    .show()
)

In [ ]:
# Warm-up 3: orderBy descending
(
    df_sales
    .groupBy('product_id')
    .agg(F.sum('quantity').alias('total_units'))
    .orderBy('total_units', ascending=False)   # highest first
    .show()
)

## Problem

Find all products where total quantity sold **exceeds 50**.  
Sort by total units **descending**.

**Expected columns:** `product_id`, `total_units_sold`

**Hints:**
- `.groupBy('product_id').agg(F.sum('quantity').alias('total_units_sold'))`
- `.filter(F.col('total_units_sold') > 50)` — PySpark's HAVING
- `.orderBy('total_units_sold', ascending=False)`

In [ ]:
# YOUR CODE HERE
df_top_products = None

# df_top_products.show()

---
# Question 3 🟡 — Regional Price & Quantity Summary

## Concept Warm-up

In [ ]:
# Warm-up 1: F.min and F.max
(
    df_sales
    .groupBy('region')
    .agg(
        F.min('unit_price').alias('min_price'),
        F.max('unit_price').alias('max_price'),
    )
    .orderBy('region')
    .show()
)

In [ ]:
# Warm-up 2: F.avg and F.round
(
    df_sales
    .groupBy('region')
    .agg(
        F.round(F.avg('unit_price'), 2).alias('avg_price'),
    )
    .orderBy('region')
    .show()
)

In [ ]:
# Warm-up 3: multiple aggregates in one .agg() call — one shuffle pass
(
    df_sales
    .groupBy('region')
    .agg(
        F.count('*').alias('num_transactions'),
        F.sum('quantity').alias('total_qty'),
    )
    .orderBy('region')
    .show()
)

## Problem

For each region produce a summary with:
- `min_price` — minimum `unit_price`
- `max_price` — maximum `unit_price`
- `avg_price` — average `unit_price` rounded to 2dp
- `total_qty` — total `quantity` sold

Order by `region` alphabetically.

**Hints:** Put all four aggregates in a single `.agg()` call. Use `F.round(F.avg(...), 2)` for the average.

In [ ]:
# YOUR CODE HERE
df_region_summary = None

# df_region_summary.show()

In [ ]:
spark.stop()
print('Spark stopped.')